In [1]:
#This is the program to get arxiv data about chinese company

In [1]:
import pprint

import arxiv
import pandas as pd 
import time
import requests
import xml.etree.ElementTree as ET

In [2]:
from dotenv import load_dotenv
import os

In [16]:
import json

In [24]:
pd.set_option('display.max_colwidth', None)

In [3]:
load_dotenv()
OPENALEX_KEY = os.environ["OPENALEX_KEY"]

In [4]:
keyword = "autonomous driving"

In [5]:
import re

In [21]:
keyword = "autonomous driving"

KEYWORD = keyword     # ここだけ差し替える
SINCE   = "2026-04-01"
Country  = "cn"
ARXIV_SRC = "S4306400194"                            # /sources?search=arxiv で確認したID

r = requests.get("https://api.openalex.org/works", params={
    "filter": ",".join([
        f"abstract.search:{KEYWORD}",
        f"primary_location.source.id:{ARXIV_SRC}",
        f"institutions.country_code:{Country}",
        f"from_publication_date:{SINCE}",
    ]),
    "sort": "publication_date:desc",
    "per_page": 25,
    "api_key": OPENALEX_KEY,
}, timeout=30)
r.raise_for_status()

In [22]:
print(r.json()["meta"])

{'count': 26, 'db_response_time_ms': 33, 'page': 1, 'per_page': 25, 'groups_count': None, 'x_query': {'oql': 'works where abstract has (autonomous driving)\n  and institution country is (cn)\n  and date >= (2026-04-01)\n  and source is (S4306400194)', 'oqo': {'get_rows': 'works', 'filter_rows': [{'column_id': 'abstract.search', 'value': 'autonomous driving', 'operator': 'has'}, {'column_id': 'authorships.institutions.country_code', 'value': 'cn'}, {'column_id': 'from_publication_date', 'value': '2026-04-01'}, {'column_id': 'primary_location.source.id', 'value': 'S4306400194'}]}, 'url': '/works?filter=abstract.search:autonomous driving,authorships.institutions.country_code:cn,from_publication_date:2026-04-01,primary_location.source.id:S4306400194&per_page=25'}, 'cost_usd': 0.001}


In [19]:
def count(filters):
    r = requests.get("https://api.openalex.org/works",
                     params={"filter": ",".join(filters), "per_page": 1,
                             "api_key": OPENALEX_KEY}, timeout=30)
    return r.json().get("meta", {}).get("count")

A = f"abstract.search:{KEYWORD}"
B = f"primary_location.source.id:{ARXIV_SRC}"
C = f"institutions.country_code:{Country}"
D = f"from_publication_date:{SINCE}"

print("日付なし:", count([A, B, C]))       # 日付が原因か
print("国なし  :", count([A, B, D]))       # 所属が原因か
print("arXivなし:", count([A, C, D]))      # Source IDが原因か
print("A+Bのみ :", count([A, B]))

日付なし: 0
国なし  : 0
arXivなし: 1133
A+Bのみ : 0


In [20]:
r = requests.get("https://api.openalex.org/sources",
                 params={"search": "arxiv", "per_page": 10,
                         "api_key": OPENALEX_KEY}, timeout=30)
for s in r.json()["results"]:
    print(s["id"], "|", s["display_name"], "|", s.get("works_count"))

https://openalex.org/S4306400194 | arXiv (Cornell University) | 3261621
https://openalex.org/S4404677083 | AI-EDU arxiv. | 9
https://openalex.org/S4210233203 | IACAPAP ArXiv | 5


In [23]:
rows = []
for w in r.json()["results"]:
    countries = sorted({c for a in w["authorships"] for c in a.get("countries", [])})
    rows.append({
        "openalex_id": w["id"],
        "title": w["title"],
        "doi": w.get("doi"),
        "publication_date": w.get("publication_date"),
        "language": w.get("language"),
        "countries": countries,
        "institutions": sorted({i["display_name"] for a in w["authorships"]
                                for i in a.get("institutions", [])}),
    })
rows

[{'openalex_id': 'https://openalex.org/W7170417272',
  'title': 'MatDiffract: A Material-Informed Automated Analysis Platform for X-ray Powder Diffraction',
  'doi': 'https://doi.org/10.48550/arxiv.2607.20880',
  'publication_date': '2026-07-23',
  'language': None,
  'countries': ['CN'],
  'institutions': ['China Spallation Neutron Source',
   'Chinese Academy of Sciences',
   'Dongguan University of Technology',
   'Henan Normal University',
   'Institute of High Energy Physics',
   'Songshan Lake Materials Laboratory',
   'Sun Yat-sen University']},
 {'openalex_id': 'https://openalex.org/W7169633445',
  'title': 'S-squared-VLA: Decoupling Semantic and Spatial Streams in Vision-Language-Action Models for Autonomous Driving',
  'doi': None,
  'publication_date': '2026-07-15',
  'language': None,
  'countries': ['CN'],
  'institutions': ['Hubei University of Technology',
   'Wuhan University of Technology']},
 {'openalex_id': 'https://openalex.org/W7169473175',
  'title': 'Hierarchical

In [27]:
def s2_get(path, params, retries=6):
    for a in range(retries):
        r = requests.get(f"https://api.semanticscholar.org/graph/v1/{path}",
                         params=params, timeout=30)
        if r.status_code == 429:
            wait = 5 * (a + 1)          # 共有プールは混みやすいので長めに待つ
            print(f"429, {wait}s待機")
            time.sleep(wait)
            continue
        return r
    return None

In [28]:
ref_cache = {}          
works = r.json()["results"]


def arxiv_id(w):
    m = re.search(r"arxiv\.(\d{4}\.\d{4,5})", w.get("doi") or "", re.I)
    if m:
        return m.group(1)
    url = (w.get("primary_location") or {}).get("landing_page_url") or ""
    m = re.search(r"arxiv\.org/abs/([^v\s?]+)", url, re.I)
    return m.group(1) if m else None

def fetch_refs(aid):
    if aid in ref_cache:
        return ref_cache[aid]
    r = s2_get(f"paper/arXiv:{aid}/references",
               {"fields": "title,year,externalIds", "limit": 500})
    if r is None:
        return None                      # リトライを使い切った
    if r.status_code == 404:
        ref_cache[aid] = []              # 未収録も記録して再取得しない
        return []
    r.raise_for_status()
    ref_cache[aid] = r.json().get("data", [])
    return ref_cache[aid]

rows = []
for i, w in enumerate(works, 1):
    aid = arxiv_id(w)
    refs = fetch_refs(aid) if aid else None
    print(f"{i}/{len(works)} {aid} refs={len(refs) if refs is not None else 'FAIL'}")

    if not refs:
        rows.append({"citing_arxiv_id": aid, "citing_title": w["title"],
                     "status": "no_refs"})
        continue
    for ref in refs:
        cp = ref.get("citedPaper") or {}
        rows.append({
            "citing_arxiv_id": aid,
            "citing_title": w["title"],
            "cited_title": cp.get("title"),
            "cited_year": cp.get("year"),
            "cited_doi": (cp.get("externalIds") or {}).get("DOI"),
            "cited_arxiv": (cp.get("externalIds") or {}).get("ArXiv"),
            "status": "ok",
        })
    time.sleep(2)                        # 429を減らすための予防的な間隔

df = pd.DataFrame(rows)

1/25 2607.20880 refs=38
2/25 2607.13926 refs=30
3/25 2607.12598 refs=53
4/25 2607.09772 refs=22
5/25 2607.04974 refs=60
6/25 2606.30537 refs=60
7/25 2606.29286 refs=43
8/25 2606.12826 refs=58
9/25 2606.08170 refs=28
10/25 2606.06014 refs=43
11/25 2605.31041 refs=27
429, 5s待機
429, 10s待機
12/25 2605.31476 refs=82
13/25 2605.30939 refs=36
14/25 2605.31572 refs=84
15/25 2605.26477 refs=36
429, 5s待機
16/25 2605.21112 refs=30
17/25 2605.18895 refs=43
18/25 2605.16859 refs=0
19/25 2605.08839 refs=83
20/25 2605.00412 refs=35
21/25 2605.00066 refs=33
22/25 2605.00907 refs=44
23/25 2604.23105 refs=60
24/25 2604.22552 refs=59
25/25 2604.07991 refs=29


In [29]:
df.head(10)

,citing_arxiv_id,citing_title,cited_title,cited_year,cited_doi,cited_arxiv,status
0,2607.20880,MatDiffract: A Material-Informed Automated Analysis Platform for X-ray Powder Diffraction,Equivariant diffusion solution for inorganic crystal structure determination from powder X-ray diffraction data,2026.0,10.1038/s41467-026-70035-9,NaN,ok
1,2607.20880,MatDiffract: A Material-Informed Automated Analysis Platform for X-ray Powder Diffraction,Machine learning-driven crystal system prediction for perovskites using augmented XRD data,2026.0,10.1016/j.engappai.2025.113247,2602.04435,ok
2,2607.20880,MatDiffract: A Material-Informed Automated Analysis Platform for X-ray Powder Diffraction,Automated phase mapping of high-throughput X-ray diffraction data encoded with domain-specific materials science knowledge,2025.0,10.1038/s41524-025-01837-6,NaN,ok
3,2607.20880,MatDiffract: A Material-Informed Automated Analysis Platform for X-ray Powder Diffraction,KAN-enhanced contrastive learning: the accelerator of crystal structure identification from XRD patterns,2025.0,10.1038/s41524-026-02015-y,2511.04055,ok
4,2607.20880,MatDiffract: A Material-Informed Automated Analysis Platform for X-ray Powder Diffraction,Dara: Automated Multiple-Hypothesis Phase Identification and Refinement from Powder X‑ray Diffraction,2025.0,10.1021/acs.chemmater.5c02820,2510.19667,ok
5,2607.20880,MatDiffract: A Material-Informed Automated Analysis Platform for X-ray Powder Diffraction,XQueryer: an intelligent crystal structure identifier for powder X-ray diffraction,2025.0,10.1093/nsr/nwaf421,NaN,ok
6,2607.20880,MatDiffract: A Material-Informed Automated Analysis Platform for X-ray Powder Diffraction,Machine learning in X-ray diffraction for materials discovery and characterization,2025.0,10.1016/j.matt.2025.102272,NaN,ok
7,2607.20880,MatDiffract: A Material-Informed Automated Analysis Platform for X-ray Powder Diffraction,Interpretable X-ray diffraction spectra analysis using confidence evaluated deep learning enhanced by template element replacement,2025.0,10.1038/s41524-025-01743-x,NaN,ok
8,2607.20880,MatDiffract: A Material-Informed Automated Analysis Platform for X-ray Powder Diffraction,Computational investigation into XRD peak broadening effects with discrete dislocation dynamics in additively manufactured 316L stainless steel,2024.0,10.1016/j.jmps.2024.105970,NaN,ok
9,2607.20880,MatDiffract: A Material-Informed Automated Analysis Platform for X-ray Powder Diffraction,Crystallographic phase identifier of a convolutional self-attention neural network (CPICANN) on powder diffraction patterns,2024.0,10.1107/S2052252524005323,NaN,ok


In [30]:
COLS = ["matched_title", "language", "countries"]

In [31]:
def lookup_openalex(title, api_key=OPENALEX_KEY):
    r = requests.get("https://api.openalex.org/works",
                     params={"filter": f"title.search:{title}",
                             "per_page": 1,
                             "api_key": api_key},      # mailtoの代わりにキーを渡す
                     timeout=30)
    if r.status_code == 429:
        return "RATE_LIMITED"
    if r.status_code != 200:
        return None
    results = r.json().get("results", [])
    if not results:
        return None
    w = results[0]
    return {
        "matched_title": w.get("title"),
        "language": w.get("language"),
        "countries": [i["country_code"] for a in w.get("authorships", [])
                      for i in a.get("institutions", []) if i.get("country_code")],
        "doi": w.get("doi"),
    }

In [33]:
cited = df[df.status == "ok"]
print("行数", len(cited))
print("DOIあり", cited.cited_doi.notna().sum())
print("ユニークDOI", cited.cited_doi.nunique())
print("必要コール数", -(-cited.cited_doi.nunique() // 100))

行数 1116
DOIあり 929
ユニークDOI 767
必要コール数 8


In [34]:
cited = df[df.status == "ok"].copy()

def norm_doi(d):
    if not d:
        return None
    return str(d).lower().replace("https://doi.org/", "").strip()

# 1. 被引用論文のメタデータを一括取得（ループの外で1回だけ）
dois = sorted({norm_doi(d) for d in cited.cited_doi.dropna()})
print("ユニークDOI", len(dois), "→ 必要コール数", -(-len(dois) // 100))

meta = {}
for i in range(0, len(dois), 100):
    batch = dois[i:i + 100]
    r = requests.get("https://api.openalex.org/works", params={
        "filter": "doi:" + "|".join(f"https://doi.org/{d}" for d in batch),
        "per_page": 100,
        "api_key": OPENALEX_KEY,
    }, timeout=60)
    if r.status_code != 200:                  # 異常時は中断して枠を守る
        print(r.status_code, r.text[:300])
        break
    for w in r.json()["results"]:
        meta[norm_doi(w.get("doi"))] = {
            "matched_title": w.get("title"),
            "language": w.get("language"),
            "countries": ",".join(sorted({c for a in w.get("authorships", [])
                                            for c in a.get("countries", [])})),
        }
    time.sleep(0.3)

# 2. 全行に付与する
key = cited.cited_doi.map(norm_doi)
for col in ["matched_title", "language", "countries"]:
    cited[col] = key.map(lambda k: (meta.get(k) or {}).get(col))

# 3. 引用元ごとに1ファイル書き出す
for i, (aid, g) in enumerate(cited.groupby("citing_arxiv_id"), 1):
    g.to_csv(f"{aid}.csv", index=False, encoding="utf-8-sig")
    print(i, aid, len(g), "行 / 国判明", (g.countries.fillna("") != "").sum())

ユニークDOI 767 → 必要コール数 8
1 2604.07991 29 行 / 国判明 12
2 2604.22552 59 行 / 国判明 25
3 2604.23105 60 行 / 国判明 35
4 2605.00066 33 行 / 国判明 12
5 2605.00412 35 行 / 国判明 9
6 2605.00907 44 行 / 国判明 16
7 2605.08839 83 行 / 国判明 58
8 2605.18895 43 行 / 国判明 33
9 2605.21112 30 行 / 国判明 23
10 2605.26477 36 行 / 国判明 17
11 2605.30939 36 行 / 国判明 29
12 2605.31041 27 行 / 国判明 11
13 2605.31476 82 行 / 国判明 28
14 2605.31572 84 行 / 国判明 29
15 2606.06014 43 行 / 国判明 21
16 2606.08170 28 行 / 国判明 18
17 2606.12826 58 行 / 国判明 46
18 2606.29286 43 行 / 国判明 31
19 2606.30537 60 行 / 国判明 28
20 2607.04974 60 行 / 国判明 19
21 2607.09772 22 行 / 国判明 15
22 2607.12598 53 行 / 国判明 34
23 2607.13926 30 行 / 国判明 11
24 2607.20880 38 行 / 国判明 31


In [35]:
cited.to_csv("cited_all_china_ar.csv", index=False, encoding="utf-8-sig")
print(len(cited))

1116
